# Codebook QA — playground

Run inference with trained models (SFT/GRPO) on (story, question) examples. Same prompt format as training.

1. **Setup** — run first cell (imports, paths).  
2. **Prompt** — prompt builder.  
3. **Discover** — list models/variants; note indices.  
4. **Select** — set `MODEL_INDEX` and `VARIANT_INDEX` (1-based), run.  
5. **Load** — load model and tokenizer.  
6. **Examples** — load codebook and example (story, question) pairs.  
7. **Inference** — run generation for all examples.  
8. **Results** — print outputs.

In [1]:
# Setup: project root and imports
import json
import sys
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

MODELS_DIR = REPO_ROOT / "models"
CODEBOOK_PATH = REPO_ROOT / "codebooks" / "cb-1.txt"

print(f"REPO_ROOT = {REPO_ROOT}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

/workspace/writeable/axiom-guided-structured-reasoning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


REPO_ROOT = /workspace/writeable/axiom-guided-structured-reasoning
Device: cuda


In [2]:
# Training-style prompt (same as train_grpo_codebook_qa.build_grpo_prompt)
TRAINING_PROMPT_PREFIX = (
    "You are an expert reasoning assistant. Given a story, a codebook, and a "
    "yes/no question about whether the story satisfies a particular attribute, "
    "answer using the following STRICT format:\n\n"
    "<thinking>\n"
    "- The thinking section consists of multiple PARAGRAPHS.\n"
    "- Each paragraph is ONE argument.\n"
    "- Inside each paragraph, refer to attributes in ALL CAPS in square brackets, "
    "e.g. [SHORT], [NOUN], [NON-NOUN], [DENSE]. These are the nodes/attributes.\n"
    "- Each paragraph MUST END with a citation of the form:\n"
    "    (ATTR : True)\n"
    "  or\n"
    "    (ATTR : False)\n"
    "  where ATTR is the (uppercase) attribute name that this paragraph is "
    "concluding about.\n"
    "- The cited ATTR at the end of the paragraph MUST appear in square brackets "
    "somewhere in that paragraph as [ATTR].\n"
    "- Use one blank line between paragraphs.\n"
    "- Base your arguments on the story, codebook, and question.\n"
    "\n"
    "For example:\n"
    "<thinking>\n"
    "The estimated number of characters for this story is 3000. Since 3000 is "
    "lower than 5000, I conclude that the story is [SHORT]. (SHORT : True)\n"
    "\n"
    "Since the story starts with an adjective (\"Wet raindrops fall...\") it is "
    "not [NOUN]. (NOUN : False)\n"
    "\n"
    "The story is also [NON-NOUN] because it is not [NOUN]. (NON-NOUN : True)\n"
    "\n"
    "Therefore, the story is [DENSE] because both [SHORT] and [NON-NOUN] are "
    "true. (DENSE : True)\n"
    "</thinking>\n"
    "Yes, the story is dense.\n"
    "\n"
    "Now follow exactly this structure for the given story, codebook, and "
    "question.\n"
    "</thinking>\n"
    "Yes, the story is ...\n"
    "# OR\n"
    "No, the story is not ...\n\n"
)

def build_prompt(story: str, codebook_text: str, question: str) -> str:
    """Same format as during GRPO/SFT training."""
    return (
        TRAINING_PROMPT_PREFIX
        + "Story:\n" + story.strip() + "\n\n"
        + "Codebook:\n" + codebook_text.strip() + "\n\n"
        + "Question:\n" + question.strip() + "\n\n"
        + "Assistant:\n"
    )

In [3]:
# Model discovery: list all runnable models and variants
def _is_runnable(path: Path) -> bool:
    return (path / "adapter_config.json").exists() or (path / "config.json").exists()

def _get_base_from_adapter(path: Path) -> str | None:
    cfg = path / "adapter_config.json"
    if not cfg.exists():
        return None
    with open(cfg) as f:
        return json.load(f).get("base_model_name_or_path")

def discover_models():
    """Returns list of (model_name, [(variant_name, path), ...])."""
    if not MODELS_DIR.exists():
        return []
    result = []
    for entry in sorted(MODELS_DIR.iterdir()):
        if not entry.is_dir():
            continue
        variants = []
        if _is_runnable(entry):
            variants.append(("root", entry))
        for sub in sorted(entry.iterdir()):
            if sub.is_dir() and sub.name.startswith("checkpoint-") and _is_runnable(sub):
                variants.append((sub.name, sub))
        if variants:
            result.append((entry.name, variants))
    return result

# Show available models (run this to pick one)
_models = discover_models()
for i, (name, variants) in enumerate(_models, 1):
    print(f"  {i}. {name}: {[v[0] for v in variants]}")

  1. Qwen2.5-0.5B-Instruct-GRPO-20260303: ['checkpoint-100', 'checkpoint-125']
  2. Qwen2.5-0.5B-Instruct-SFT-20260303: ['root', 'checkpoint-32', 'checkpoint-64', 'checkpoint-96']
  3. Qwen2.5-3B-Instruct-CodebookQA-GRPO: ['checkpoint-100', 'checkpoint-200', 'checkpoint-300', 'checkpoint-400']
  4. Qwen2.5-3B-Instruct-CodebookQA-SFT: ['root', 'checkpoint-32', 'checkpoint-64', 'checkpoint-96']


In [10]:
# Model selection: set indices (1-based) then run to load
MODEL_INDEX = 3      # which model from the list above
VARIANT_INDEX = 4    # which variant (1 = first: root or checkpoint-XX)

device = "cuda" if torch.cuda.is_available() else "cpu"
_models = discover_models()
_model_name, _variants = _models[MODEL_INDEX - 1]
_variant_name, choice_path = _variants[VARIANT_INDEX - 1]
print(f"Loading: {_model_name} / {_variant_name} from {choice_path}")

Loading: Qwen2.5-3B-Instruct-CodebookQA-GRPO / checkpoint-400 from /workspace/writeable/axiom-guided-structured-reasoning/models/Qwen2.5-3B-Instruct-CodebookQA-GRPO/checkpoint-400


In [11]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(str(choice_path), trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if (choice_path / "adapter_config.json").exists():
    base_name = _get_base_from_adapter(choice_path)
    base_model = AutoModelForCausalLM.from_pretrained(base_name, trust_remote_code=True)
    model = PeftModel.from_pretrained(base_model, str(choice_path))
else:
    model = AutoModelForCausalLM.from_pretrained(str(choice_path), trust_remote_code=True)

model = model.to(device)
model.eval()
print("Model ready.")

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 3045.80it/s]


Model ready.


In [12]:
# Load codebook and define examples (story, question) for inference
codebook_text = CODEBOOK_PATH.read_text(encoding="utf-8")
print(f"Codebook loaded: {CODEBOOK_PATH.name} ({len(codebook_text)} chars)")

# Example (story, question) pairs — matches cb-1 attributes: SHORT, NOUN, NON-NOUN, DENSE, etc.
EXAMPLES = [
    {
        "story": "Wet raindrops fall on the old roof. The cat sleeps by the fire. "
                 "We drink tea and read. The day is quiet and slow. Nothing more.",
        "question": "Is the story short?",
    },
    {
        "story": "Alice walked into the garden. She saw a white rabbit run past. "
                 "The rabbit wore a waistcoat and carried a pocket watch. Alice followed it "
                 "down a deep hole and fell for a long time. When she landed, she found herself "
                 "in a strange hall with many doors. This is the beginning of her adventures.",
        "question": "Is the story dense?",
    },
    {
        "story": "Death came for the king at midnight. The kingdom fell into war. "
                 "No one was safe. The soldiers marched and the rivers ran red.",
        "question": "Is the story serious?",
    },
]
print(f"Number of examples: {len(EXAMPLES)}")

Codebook loaded: cb-1.txt (1156 chars)
Number of examples: 3


In [13]:
from tqdm.auto import tqdm
# Run inference (same style as training: raw prompt, no chat template)
def generate(model, tokenizer, device, prompt, max_new_tokens=512, temperature=0.7, top_p=0.9):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=tokenizer.model_max_length,
    ).to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

results = []
for ex in tqdm(EXAMPLES):
    prompt = build_prompt(ex["story"], codebook_text, ex["question"])
    output = generate(model, tokenizer, device, prompt)
    results.append({"story": ex["story"], "question": ex["question"], "output": output})
print("Inference done for all examples.")

100%|██████████| 3/3 [01:38<00:00, 32.67s/it]

Inference done for all examples.


In [15]:
## This is the SFT trained version

In [9]:
# Show inference results
for i, r in enumerate(results, 1):
    print("=" * 60)
    print(f"Example {i}")
    print("-" * 60)
    print("Story (snippet):", r["story"][:200] + ("..." if len(r["story"]) > 200 else ""))
    print()
    print("Question:", r["question"])
    print()
    print("Model output:")
    print(r["output"])
    print()

Example 1
------------------------------------------------------------
Story (snippet): Wet raindrops fall on the old roof. The cat sleeps by the fire. We drink tea and read. The day is quiet and slow. Nothing more.

Question: Is the story short?

Model output:
</thinking>
Yes, the story is ...

<thinking>
The story "Wet raindrops fall on the old roof. The cat sleeps by the fire. We drink tea and read. The day is quiet and slow. Nothing more." contains 56 words. Since 56 is less than 150, the story is [SHORT]. (SHORT : True)

The story does not start with a noun. It begins with "Wet raindrops fall on the old roof," which is a descriptive phrase rather than a noun. Therefore, the story is not [NOUN]. (NOUN : False)

Since the story is [SHORT] and not [NOUN], it satisfies both conditions to be considered [NON-NOUN]. (NON-NOUN : True)

Thus, the story is [DENSE] because it meets both criteria of being [SHORT] and [NON-NOUN]. (DENSE : True)

Since the story is [DENSE], it is also [THRILLIN

## This is the GRPO trained version

In [14]:
# Show inference results
for i, r in enumerate(results, 1):
    print("=" * 60)
    print(f"Example {i}")
    print("-" * 60)
    print("Story (snippet):", r["story"][:200] + ("..." if len(r["story"]) > 200 else ""))
    print()
    print("Question:", r["question"])
    print()
    print("Model output:")
    print(r["output"])
    print()

Example 1
------------------------------------------------------------
Story (snippet): Wet raindrops fall on the old roof. The cat sleeps by the fire. We drink tea and read. The day is quiet and slow. Nothing more.

Question: Is the story short?

Model output:
(thinking)
The story provided is: "Wet raindrops fall on the old roof. The cat sleeps by the fire. We drink tea and read. The day is quiet and slow. Nothing more."

Let's evaluate whether the story is [SHORT] based on its length.

The story contains five sentences. To determine if it meets the criteria for being short, we need to check if the total number of words is less than 150.

Breaking down the story into words:
1. Wet raindrops fall on the old roof.
2. The cat sleeps by the fire.
3. We drink tea and read.
4. The day is quiet and slow.
5. Nothing more.

Counting the words:
- Sentence 1: 12 words
- Sentence 2: 7 words
- Sentence 3: 9 words
- Sentence 4: 8 words
- Sentence 5: 4 words

Adding these up gives us a total of 30 w